# Closed action spaces

Jev picks from moves you already decided are legal. It cannot invent a fifth button. A question that is not a device command falls through to a normal reply, which we only label here.


In [ ]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


## 64. Pick a legal move

Two trail states. In the dangerous one, retreat wins if it is legal, even if walking would also make progress.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    game = load_json("game.json")
    questions = {
        "move": Choice(instructions="Given `state`, which move best advances `objective`?", criteria=game["legal_moves"]),
        "in_danger": Noul(instructions="Is the agent in immediate danger in `state`?"),
    }
    for label, state in (("safe", game["state_safe"]), ("danger", game["state_danger"])):
        response = ask({"state": state, "objective": game["objective"]}, questions)
        show(response)
        if response.nouls["in_danger"].noul > 0.65 and "retreat" in game["legal_moves"]:
            move = "retreat"
        else:
            move = response.choices["move"].choice
        print(label, "->", move)


**What you should see.** The low creek should walk or wait, not retreat. The creek over the trail should retreat.


## 65. Only act on a real device command

Devices, actions, and rooms are closed lists. A store-hours question is not a command.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    home = load_json("devices.json")
    questions = {
        "device": Choice(instructions="Which device does `command` target?", criteria=home["devices"]),
        "action": Choice(instructions="What action does `command` request?", criteria=home["actions"]),
        "room": Choice(instructions="Which room does `command` refer to?", criteria=home["rooms"]),
        "is_command": Noul(instructions="Is `command` an instruction to control a device, rather than a question?"),
    }
    for command in home["commands"]:
        response = ask({"command": command}, questions)
        show(response)
        device = response.choices["device"]
        action = response.choices["action"]
        if response.nouls["is_command"].noul < 0.55:
            route = "conversation"
        elif device.choice == "none" or action.choice == "none" or min(device.confidence, action.confidence) < 0.55:
            route = "ask_which"
        else:
            route = "execute %s %s in %s" % (action.choice, device.choice, response.choices["room"].choice)
        print(command, "->", route)


**What you should see.** The kettle sentence should execute turn_on kettle in the kitchen. The store-hours question should stay a conversation.
